In [20]:
import oracledb
import pandas as pd
import torch
import random
import json
from datetime import datetime

oracledb.init_oracle_client(lib_dir=r"D:\\instantclient_23_9")

conn = oracledb.connect(
    user="adsql",          # 사용자명
    password="oracle_4U",      # 비밀번호
    dsn="localhost:1521/xe" # 접속 정보 (SQL Developer와 동일)
)
cur = conn.cursor()

In [21]:
from prediction_all import *

In [22]:
optim_molecule = pd.read_csv('optim_molecule.csv')
optim_molecule = optim_molecule.iloc[0:0]

checkpoint = torch.load('D:\minimax\minimax_backend\model\\for_predict_file\model_checkpoint.pt', map_location=device, weights_only=False)
token2id = checkpoint['token2id']
id2token = checkpoint['id2token']

with open("D:\minimax\minimax_backend\model\\for_predict_file\\vocab.json", "r", encoding="utf-8") as f:
	data = json.load(f)   # JSON → 파이썬 딕셔너리/리스트 변환

data = token2id

# 최적화 버튼을 눌렀을 때
mode = 'user_diy'

# 1. 만약 이게 user가 입력해서 만든 분자를 최적화하려는 거면
if mode == 'user_diy':
	df = pd.read_csv('user_generative.csv')
	col_start = 'U'
else: # 버튼 누른거면
	df = pd.read_csv('disease_generative.csv')
	col_start = 'D'

return_value = {}

choose_optim = f'{col_start}NEW_MOLECULE3' # 이건 사용자가 클릭한 분자의 name
will_opt = df[df[f'{col_start}NEW_NAME'] == choose_optim].to_dict(orient='records')[0]
orig_features = will_opt

max_iter = 1
best_fitness = 10
iteration = 0
gen_size = 4

best_results = []
after_optim_mol = []
target = orig_features[f'{col_start}NEW_CANOSMILES']

mol = Chem.MolFromSmiles(target)

In [23]:
def predict_feature(smiles):
	mol = Chem.MolFromSmiles(smiles) 
	new_optim_predict = [will_opt[f'{col_start}NEW_NAME'],
							f'ONEW_MOLECULE{str(len(optim_molecule))}',
							smiles,
							smiles_to_svg_base64(smiles)]

	new_optim_predict += isit_available_medicine(mol) # [molecule_weight, logp, qed, hbd, hba]
	pki = list(predict_pKi(smiles))
	pkd = list(predict_pKd(smiles))
	toxic = [toxic_predict(smiles)]
	new_optim_predict = new_optim_predict + pki + pkd + toxic + [datetime.now().strftime('%Y-%m-%d %H:%M:%S')] + [0]
	
	optim_molecule.loc[len(optim_molecule)] = new_optim_predict
	return dict(zip(list(optim_molecule.columns),new_optim_predict))

def randch():
	rand_mole = random.sample([i for i in list(data.keys()) if i not in ['[EOS]','[SOS]','[PAD]']],1)
	return rand_mole[0] # 여기서 랜덤으로 하나 뽑기

# make_random_chrono: 무작위 해 하나 생성
def make_random_chrono(target): # 원조 분자의 일부를 삭제하거나 더하는 걸 랜덤으로 수행함
	chrono = list(sf.split_selfies(sf.encoder(target)))
	for i in range(3):
		x = random.randint(0,1)
		if x == 0:
			remove_molecule = random.randint(0, len(chrono)-1)
			chrono.remove(chrono[remove_molecule])
		else:
			chrono.insert(random.randint(0,len(chrono)),randch())

	return sf.decoder(''.join(chrono))

# make_random_generation: 초기 해집단 생성(= 해를 gen_size개만큼 생성) = 5개
def make_random_generation(target):
	return [make_random_chrono(target) for _ in range(gen_size)]

# get_fitness: 적합도 계산 
# 분자 smiles에서는 기존 독성 예측 등의 값과 해집단의 예측값을 비교해서 차이가 많이 나면 좋은거로
def get_fitness(smiles, orig_features):
	mol = Chem.MolFromSmiles(smiles) 
	features = predict_feature(smiles)  # 새로운 분자 특성 예측      

	# Lipinski 기준 만족 여부 (True=1, False=0)
	lip_pass = (features['ONEW_MOL_WEIGHT'] <= 500 and
			features['ONEW_LOGP'] <= 5 and
			features['ONEW_HBD'] <= 5 and
			features['ONEW_HBA'] <= 10)

	# 간단한 스코어링: 낮은 tox → 1/tox, 높은 qed, 높은 pKi, 높은 pKd
	score = sum([-(features['ONEW_TOXIC'] - orig_features[f'{col_start}NEW_TOXIC']),
		(features['ONEW_QED'] - orig_features[f'{col_start}NEW_QED']),
		(features['ONEW_PKI'] - orig_features[f'{col_start}NEW_PKI']),
		(features['ONEW_PKD'] - orig_features[f'{col_start}NEW_PKD'])])
	# Lipinski 조건을 모두 만족하면 보너스
	if lip_pass:
		score += 10
	return score

def make_roulette(generation):
	fitnesses = [get_fitness(c,orig_features) for c in generation] # 적합도 계산
	
	prev_value = 0.0
	roulette = [0.0]
	for f in fitnesses:
		value = float(f / sum(fitnesses))
		roulette.append(prev_value + value)
		prev_value += value
	
	return roulette

# selection: 룰렛 휠 선택 연산 #
def selection(chronos, roulette):
	selected_chrono = None
	dart = random.random()  # 다트 던지기 EX) 0.27

	# 룰렛에서 해 선택
	for idx in range(1, len(roulette)):
		if dart < roulette[idx]: # dart보다 룰렛 적합도가 높으면
			selected_chrono = chronos[idx-1] # 그 적합도가 높은 chrono를 선택함
			break
	
	return selected_chrono

# crossover: 1점 교차 연산 -> 그냥 부모 2개 랜덤으로 잘라서 이어 붙이는거
def crossover(ca, cb):
	ca2 = list(sf.split_selfies(sf.encoder(ca)))
	cb2 = list(sf.split_selfies(sf.encoder(cb)))
	cross_point = random.randint(1, min(len(ca2),len(cb2))-1)     
	offspring = ca2[:cross_point] + cb2[cross_point:]  
	return offspring

# mutation: 변이 연산 #
def mutation(chrono):
	mutated_chrono = chrono # 이건 
	propability = 0.1      # 변이 확률 0.03
	good_token = ['[O]', '[N]', '[C]', '[=O]']
	branch_list = [i for i in mutated_chrono if 'Branch' in i]

	if random.random() < propability:
		case = random.randint(0,1)
		if branch_list and case == 0:
			mutated_chrono.remove(random.choice(branch_list))
		else:
			mutated_chrono.append(random.choice(good_token))

	# 변이된 문자열 반환
	return sf.decoder(''.join(mutated_chrono))

# sort_generation: 적합도를 기반으로 해집단 정렬
def sort_generation(generation):
	fitnesses = [get_fitness(c,orig_features) for c in generation]
	sorted_gen = [c for _, c in sorted(zip(fitnesses, generation))]
	return sorted_gen # 적합도 낮은 거부터 튀어나옴

# make_offsprings: 부모 세대로부터 자식 세대 생성 #
def make_offsprings(generation):
	ggap = 0.8  # 세대차
	sorted_gen = sort_generation(generation)    # 정렬된 해집단
	n_parents = int(gen_size * (1.0 - ggap))    # 남겨놓을 부모 해의 개수는 3개
	offsprings = sorted_gen[-n_parents:]         # 우수한 부모 해 그대로 남기기 (적합도 높은 상위 3개)
	roulette = make_roulette(sorted_gen)        # 룰렛 생성

	# 남은 수만큼 자식해 생성 후 대치
	for i in range(gen_size - n_parents):
		ca = selection(sorted_gen, roulette)    # 부모해 선택 1 (dart 던져서 적합도가 높은 거1, 근데 dart가 랜덤이라 걍 랜덤으로 추출하는듯)
		cb = selection(sorted_gen, roulette)    # 부모해 선택 2 (dart 던져서 적합도가 높은 거2)
		offspring = crossover(ca, cb)           # 교차
		offspring = mutation(offspring)         # 변이
		offsprings.append(offspring)

	return offsprings

# get_best_chrono: 가장 우수한 해와 그 적합도 반환 #    
def get_best_chrono(chronos):
	fitnesses = [get_fitness(c,orig_features) for c in chronos]
	best_fitness = max(fitnesses)
	best_idx = fitnesses.index(best_fitness)
	return chronos[best_idx], best_fitness

In [42]:
# 종료 조건 만족(최적해 발견) 시까지 반복
generation = make_random_generation(target) # chronons

while best_fitness <= 20 and iteration < max_iter:
	best_result = []
	iteration += 1
	best_chrono, best_fitness = get_best_chrono(generation)
	print('Gen', iteration, '---', 'Best:', best_chrono, 'fitness:', best_fitness)
	best_result += [iteration, best_chrono, best_fitness]
	generation = make_offsprings(list(optim_molecule.sort_values('ONEW_OPTIM_TIME',ascending=False).iloc[:4]['ONEW_CANOSMILES']))
	best_results.append(best_result)

optim_molecule = optim_molecule.drop_duplicates('ONEW_CANOSMILES')
optim_molecule['ONEW_IMAGE_BASE64'] = optim_molecule['ONEW_IMAGE_BASE64'].apply(lambda x: x.read() if hasattr(x, 'read') else x)
optim_molecule.loc[optim_molecule['ONEW_CANOSMILES'].isin([i[1] for i in best_results]),'ONEW_BEST'] = 1
optim_molecule.to_csv('optim_molecule.csv',index=False)

return_value['best_result_smiles'] = optim_molecule[optim_molecule['ONEW_BEST'] == 1]['ONEW_NAME'].iloc[0]
return_value['col_start'] = col_start
return_value['after_optim_mols'] = list(optim_molecule['ONEW_NAME'])

In [43]:
return_value

{'best_result_smiles': 'ONEW_MOLECULE0',
 'col_start': 'U',
 'after_optim_mols': ['ONEW_MOLECULE0',
  'ONEW_MOLECULE1',
  'ONEW_MOLECULE2',
  'ONEW_MOLECULE3']}